# Code Lab: Imbalanced Data & PCA

In this lab, we will explore both portions of Session 7: Handling highly skewed target variables using oversampling, and reducing dimensionality using Principal Component Analysis.

In [ ]:
# Step 1: Install require dependencies
!pip install imbalanced-learn scikit-learn pandas numpy matplotlib seaborn

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Machine Learning libraries
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.decomposition import PCA
from imblearn.over_sampling import SMOTE

print("Libraries imported successfully!")

### Step 2: The Imbalance Problem
We generate a fake dataset of 10,000 transactions. Only 2% are Fraud (Class 1).

In [ ]:
# weights=[0.98, 0.02] creates the huge imbalance
X, y = make_classification(n_samples=10000, n_features=10, n_classes=2, weights=[0.98, 0.02], random_state=42)

print(f"Normal Transactions: {sum(y==0)}")
print(f"Fraud Transactions: {sum(y==1)}")

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

model = LogisticRegression()
model.fit(X_train, y_train)
preds = model.predict(X_test)

print("\n--- Classification Report (NO SMOTE) ---")
print(classification_report(y_test, preds))

**Notice the Warning:** The model achieves 98% overall accuracy, but the *Recall* for Class 1 (Fraud) is extremely low. It missed the vast majority of the fraud!

### Step 3: SMOTE (Synthetic Minority Over-sampling Technique)
We will synthesize thousands of fake fraud examples to balance the training data.

In [ ]:
smote = SMOTE(random_state=42)

# Apply SMOTE *ONLY* to the training data. Never to the testing data!
X_train_smote, y_train_smote = smote.fit_resample(X_train, y_train)

print(f"Normal Transactions after SMOTE: {sum(y_train_smote==0)}")
print(f"Fraud Transactions after SMOTE: {sum(y_train_smote==1)}")

model_smote = LogisticRegression()
model_smote.fit(X_train_smote, y_train_smote)
preds_smote = model_smote.predict(X_test)

print("\n--- Classification Report (WITH SMOTE) ---")
print(classification_report(y_test, preds_smote))

**Wow!** Overall accuracy dropped slightly, but our Class 1 Recall skyrocketed. We are now catching the fraud much more effectively.

### Step 4: PCA (Principal Component Analysis)
Our dataset currently has 10 features (dimensions), making it biologically impossible for humans to visualize clearly. We will use PCA to mathematically "crush" 10 dimensions down to 2 dimensions so we can plot it on a standard graph.

In [ ]:
pca = PCA(n_components=2)
X_pca = pca.fit_transform(X_train_smote)

print(f"Original shape: {X_train_smote.shape}")
print(f"PCA shape: {X_pca.shape}")

plt.figure(figsize=(10, 6))
# Plot Normal (Class 0) mostly transparent
plt.scatter(X_pca[y_train_smote==0, 0], X_pca[y_train_smote==0, 1], color='blue', alpha=0.1, label='Normal')
# Plot Fraud (Class 1) bright red
plt.scatter(X_pca[y_train_smote==1, 0], X_pca[y_train_smote==1, 1], color='red', alpha=0.3, label='Fraud/Synthetic')

plt.title("PCA: 10D Data Flattened to 2D")
plt.xlabel("Principal Component 1")
plt.ylabel("Principal Component 2")
plt.legend()
plt.show()